# MetroLab · 03 — Modelo, predicción y datos del tablero

Reemplaza `04_MODELO_GPU` (pasos 10 a 12). Entrada: `v3/tabla_modelo` (cuaderno 02).

**Modos** (celda 0.1)

| Modo | Cuándo | Máquina | Qué hace |
|---|---|---|---|
| `"actualizar"` | Cada vez que hay datos nuevos | **CPU** (no gasta GPU) | Carga el modelo guardado, predice el último día y arma los CSV del tablero |
| `"reentrenar"` | La primera vez, una vez al mes, o si el error sube | **GPU** (A100 o L4) | Histórico, muestra, entrenamiento, evaluación, guarda el modelo y además hace todo lo de "actualizar" |

El modelo queda en `03_MODELACION/v3/modelos/actual/` (con su histórico y la lista de variables) y una copia con fecha en `modelos/historial/`. Así ya no se pierde al cerrar Colab.

**Salida**: un `.zip` con los CSV del tablero. En modo actualizar trae 5 archivos (los que cambian cada día); en modo reentrenar, 11.

## 0. Parámetros, Spark y Azure

In [ ]:
# ============================================================
# 0.1 Parámetros
# ============================================================
MODO = "actualizar"          # "actualizar" (CPU) | "reentrenar" (GPU)

# --- Solo para reentrenar ---
# Mismos cortes de antes (entrenamiento mar–jul, validación ago, prueba sep en adelante).
# Con None se usan cortes móviles: prueba = últimos 28 días, validación = los 28 anteriores.
FIN_ENTRENAMIENTO = "2026-07-31"
FIN_VALIDACION    = "2026-08-31"
PROBAR_VARIANTES  = False    # True: compara 4 variantes (≈4× más tiempo). False: error absoluto con pesos (la ganadora)
FRAC_FLUIDO = 0.10           # fracción de filas fluidas en la muestra (llevan peso 10)
SEMILLA = 42

# --- Tablero ---
HORA_DESDE, HORA_HASTA = 5, 21    # horas del día que se muestran
MAX_MB_DIA = 7.5                  # si el archivo del día pasa de esto, se empieza más tarde

assert MODO in ("actualizar", "reentrenar")
print("Modo:", MODO)

In [ ]:
# ============================================================
# 0.2 Spark y Azure (misma configuración de 00_Conexion_Azure_PySpark)
# ============================================================
import os, sys, subprocess, time, json, shutil, gc, glob
import importlib.metadata
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
import psutil

# ---- 1. Entorno: Colab (con Google Drive) o GitHub Actions ----
try:
    from google.colab import userdata
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def secreto(*nombres):
    """Primer secreto que exista: Secretos de Colab (llave) o variables de entorno (GitHub Actions)."""
    for n in nombres:
        try:
            v = userdata.get(n) if EN_COLAB else os.environ.get(n)
        except Exception:
            v = None
        if v:
            return v
    raise ValueError(f"Falta el secreto {nombres[0]} (revisa la llave de Secretos de Colab)")

CONTENT = "/content" if EN_COLAB else os.path.abspath("trabajo")
os.makedirs(CONTENT, exist_ok=True)

PROJECT_DIR = None                               # carpeta del proyecto en Google Drive
if EN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Waze_GD"
    if os.path.isdir(PROJECT_DIR):
        PATH_NOTEBOOKS, PATH_SALIDAS, PATH_CONFIG = (f"{PROJECT_DIR}/{c}" for c in ("Notebooks", "Salidas", "Config"))
        os.makedirs(PATH_SALIDAS, exist_ok=True)
        print("✅ Google Drive conectado · proyecto:", PROJECT_DIR)
    else:
        print("⚠️ No encontré", PROJECT_DIR, "(se sigue sin Drive)")
        PROJECT_DIR = None

# ---- 2. Recursos (Colab Pro: 8 CPU y ~51 GB → 8 núcleos y ~30 GB para Spark) ----
CPU_SPARK = os.cpu_count()
RAM_GB = psutil.virtual_memory().total / 1024**3
MEMORIA_SPARK = f"{max(4, int(RAM_GB * (0.45 if MODO == "reentrenar" else 0.6)))}g"
PARTICIONES_SHUFFLE = CPU_SPARK * 8               # 8 CPU → 64, como en el script de conexión
TMP_LOCAL = f"{CONTENT}/tmp_metrolab"             # temporales en el disco local (mucho más rápido que Azure)
os.makedirs(TMP_LOCAL, exist_ok=True)

# ---- 3. Conector Hadoop-Azure ----
HADOOP_VERSION = "3.4.1"
AZURE_PACKAGE = f"org.apache.hadoop:hadoop-azure:{HADOOP_VERSION}"
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {AZURE_PACKAGE} pyspark-shell"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"        # evita conflictos de red de Spark en Colab
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ---- 4. Spark: reutiliza la sesión si ya existe ----
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F, types as T
import numpy as np
import pandas as pd

spark = SparkSession.getActiveSession()
if spark is not None:
    print("✅ Se reutiliza la sesión Spark existente")
else:
    spark = (
        SparkSession.builder
        .appName("MetroLab_03_Modelo")
        .master(f"local[{CPU_SPARK}]")
        .config("spark.driver.memory", MEMORIA_SPARK)
        .config("spark.driver.maxResultSize", "2g")
        .config("spark.sql.shuffle.partitions", str(PARTICIONES_SHUFFLE))
        .config("spark.default.parallelism", str(CPU_SPARK * 2))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.local.dir", f"{TMP_LOCAL}/spark")
        .getOrCreate()
    )
    print("✅ Nueva sesión Spark iniciada")
spark.sparkContext.setLogLevel("WARN")

# Ajustes de la línea MetroLab (se aplican también si la sesión ya existía)
for k, v in {
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    # Un solo reloj: 'ts_s' y 'v' son HORA DE BOGOTÁ guardada como si fuera UTC;
    # con la sesión en UTC, Spark muestra esas horas tal cual, sin correrlas 5 horas
    "spark.sql.session.timeZone": "UTC",
    "spark.sql.sources.partitionOverwriteMode": "dynamic",              # reescribe solo las fechas tocadas
    "spark.sql.sources.partitionColumnTypeInference.enabled": "false",  # 'fecha' se lee como texto
    "spark.sql.execution.arrow.pyspark.enabled": "true",
}.items():
    spark.conf.set(k, v)

# ---- 5. Autenticación Azure ADLS Gen2 ----
account    = secreto("AZURE_STORAGE_ACCOUNT", "storage_account_name")
key        = secreto("AZURE_STORAGE_KEY", "storage_account_key")
filesystem = secreto("AZURE_FILE_SYSTEM", "file_system_name")
endpoint   = f"{account}.dfs.core.windows.net"
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set(f"fs.azure.account.auth.type.{endpoint}", "SharedKey")
hconf.set(f"fs.azure.account.key.{endpoint}", key)
del key

BASE      = f"abfss://{filesystem}@{endpoint}/Waze_feed_tvt_parquet"
RUTA_JAMS = f"{BASE}/salidas_intermedias/02_ASIGNACION_ESPACIAL/jams_autosur_json"
SALIDA    = f"{BASE}/salidas_intermedias/03_MODELACION"
V3        = f"{SALIDA}/v3"                   # todo lo nuevo queda aquí; lo anterior no se toca

OFFSET_S  = 5 * 3600     # Bogotá = UTC − 5 (Colombia no tiene horario de verano)
VENTANA_S = 300          # ventanas de 5 minutos

# ---- Utilidades de archivos ----
Path = spark._jvm.org.apache.hadoop.fs.Path
def fs_de(ruta):  return Path(ruta).getFileSystem(hconf)
def existe(ruta): return fs_de(ruta).exists(Path(ruta))
def borrar(ruta):
    if existe(ruta):
        fs_de(ruta).delete(Path(ruta), True)
def local(nombre): return f"file://{TMP_LOCAL}/{nombre}"
def guardar_local(df, nombre):
    """Escribe en el disco de Colab y vuelve a leer: corta el plan de Spark y evita recalcular."""
    df.write.mode("overwrite").parquet(local(nombre))
    return spark.read.parquet(local(nombre))
def fechas_de(ruta):
    """Fechas guardadas en una tabla partida por 'fecha' (solo lista carpetas, no lee datos)."""
    if not existe(ruta):
        return []
    return sorted(s.getPath().getName().split("=", 1)[1] for s in fs_de(ruta).listStatus(Path(ruta))
                  if s.isDirectory() and s.getPath().getName().startswith("fecha="))
def fecha_de(col_segundos):
    return F.date_format(F.timestamp_seconds(col_segundos), "yyyy-MM-dd")

# ---- 6. Validación de conexión y rutas ----
print("\n=== CONFIGURACIÓN SPARK ===")
print("Versión:", spark.version, "· Master:", spark.sparkContext.master,
      "· Memoria:", spark.sparkContext.getConf().get("spark.driver.memory", MEMORIA_SPARK),
      "· Particiones:", spark.conf.get("spark.sql.shuffle.partitions"))
print("\n=== VALIDACIÓN DE CONEXIÓN AZURE ===")
rutas_ok = {}
for nombre, ruta in [("Waze_feed_tvt_parquet", BASE), ("02_ASIGNACION_ESPACIAL/jams_autosur_json", RUTA_JAMS),
                     ("03_MODELACION", SALIDA)]:
    rutas_ok[nombre] = existe(ruta)
    print(("✅ OK" if rutas_ok[nombre] else "❌ NO EXISTE"), "|", nombre)
if not all(rutas_ok.values()):
    raise FileNotFoundError("Revisa las rutas que no existen (o las credenciales de Azure)")
print("✅ CONEXIÓN AZURE VALIDADA · salidas nuevas en:", V3)

In [ ]:
# ============================================================
# 0.3 XGBoost, variables y funciones comunes
# ============================================================
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost as xgb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
version_xgb = tuple(int(x) for x in xgb.__version__.split(".")[:2])
PARAMS_DISPOSITIVO = ({"tree_method": "hist", "device": "cuda" if GPU else "cpu"} if version_xgb >= (2, 0)
                      else {"tree_method": "gpu_hist" if GPU else "hist"})
print("XGBoost", xgb.__version__, "· GPU:", GPU)
if MODO == "reentrenar" and not GPU:
    print("Aviso: no hay GPU. Entrenar en CPU funciona, pero tarda mucho más (Entorno de ejecución → Cambiar tipo).")

RUTA_TM = f"{V3}/tabla_modelo"
MODELOS = f"{V3}/modelos"
ACTUAL  = f"{MODELOS}/actual"
CARPETA = f"{CONTENT}/tablero"
os.makedirs(CARPETA, exist_ok=True)
for f_ in os.listdir(CARPETA):
    os.remove(f"{CARPETA}/{f_}")

HS = ["15", "30", "60"]
MINUTOS = {"15": 15, "30": 30, "60": 60}
FEATURES_BASE = [
    "sentido_n", "orden_seg", "abs_ini_m", "longitud_m", "vel_libre_ref_kmh", "vl_respaldo", "segmento_peaje",
    "vel_kmh", "cobertura", "retraso_s", "indice_vel", "nivel_max", "n_capturas",
    "vel_l1", "vel_l2", "vel_l3", "vel_l6", "vel_l12", "vel_ayer", "vel_semana",
    "vel_media_30", "vel_min_30", "delta_15",
    "vel_adelante_500", "cob_adelante_500", "vel_adelante_l3", "vel_adelante_1000",
    "vel_atras_500", "cob_atras_500", "dist_trancon_adelante_m",
    "cierre_persistente", "cierre_puntual", "cierre_cerca_1km", "hay_peaje",
    "hora_decimal", "hora_sin", "hora_cos", "dia_semana", "dia_sin", "dia_cos",
    "fin_de_semana", "festivo", "dia_habil", "puente", "retorno_puente",
    "vispera_festivo", "semana_santa", "dia_pago",
]
TIPICAS   = [f"tipica_{h}" for h in HS]
OBJETIVOS = [f"y_{h}_s" for h in HS]
TEXTO = ("sentido", "FROM_TO", "fecha", "conjunto", "estado")

def dia_de(c):    return ((F.floor(c / 86400) + 3) % 7).cast("int")     # 0 = lunes
def franja_de(c): return F.floor((c % 86400) / 300).cast("int")         # 0 … 287

def agregar_tipica(df, hist):
    """Velocidad típica del segmento en t + h (perfil histórico) para cada horizonte."""
    for h in HS:
        vh = F.col("v") + MINUTOS[h] * 60
        k = hist.select("sentido", "FROM_TO", F.col("dow").alias(f"dow_{h}"), F.col("slot").alias(f"slot_{h}"),
                        F.col("vel_tipica").alias(f"tipica_{h}"))
        df = (df.withColumn(f"dow_{h}", dia_de(vh)).withColumn(f"slot_{h}", franja_de(vh))
                .join(F.broadcast(k), ["sentido", "FROM_TO", f"dow_{h}", f"slot_{h}"], "left")
                .drop(f"dow_{h}", f"slot_{h}"))
    return df

def a_numeros(df):
    for c, t in df.dtypes:
        if t == "boolean":
            df = df.withColumn(c, F.col(c).cast("int"))
    return df.withColumn("sentido_n", (F.col("sentido") == "Occidente - Oriente").cast("int"))

def a_pandas(df, cols, nombre):
    """Spark → disco de Colab (float32) → pandas, sin juntar todo en la memoria de Spark."""
    cols = [c for c in dict.fromkeys(cols) if c in df.columns]
    df.select(*[F.col(c) if c in TEXTO or c == "v" else F.col(c).cast("float") for c in cols]) \
      .write.mode("overwrite").parquet(local(nombre))
    pdf = pq.read_table(f"{TMP_LOCAL}/{nombre}").to_pandas()
    pdf["v"] = pdf["v"].astype("int64")
    return pdf

class Modelo:
    """Un booster de XGBoost con su número de árboles (el de la mejor iteración)."""
    def __init__(self, bst, n_arboles):
        self.bst, self.n = bst, int(n_arboles)
    def predict(self, X):
        return self.bst.inplace_predict(np.asarray(X, dtype=np.float32), iteration_range=(0, self.n),
                                        validate_features=False)

def mae(real, pred, w):
    real, pred, w = np.asarray(real, float), np.asarray(pred, float), np.asarray(w, float)
    m = ~(np.isnan(real) | np.isnan(pred))
    return float(np.average(np.abs(real[m] - pred[m]), weights=w[m])) if m.any() else np.nan

tm = spark.read.parquet(RUTA_TM)
FECHAS = fechas_de(RUTA_TM)
vl_act = spark.read.parquet(f"{V3}/vel_libre_segmento").filter("activo")
print(f"Tabla de modelado: {len(FECHAS)} fechas ({FECHAS[0]} a {FECHAS[-1]})")

## R. Reentrenar (solo en modo `"reentrenar"`)

Si el modo es `"actualizar"`, estas celdas se saltan solas.

**R1 · Conjuntos, histórico y muestra.** El *histórico* (perfil típico) es el promedio de velocidad por segmento × día de la semana × franja de 5 min, **solo con entrenamiento**. La *muestra* conserva todas las filas con trancón ahora o en el futuro y el 10 % de las demás, con peso 10.

In [ ]:
# ============================================================
# R1 Conjuntos, histórico y muestra
# ============================================================
if MODO == "reentrenar":
    t0 = time.time()
    if FIN_ENTRENAMIENTO is None or FIN_VALIDACION is None:
        ultima = pd.Timestamp(FECHAS[-1])
        FIN_VALIDACION    = str((ultima - pd.Timedelta(days=28)).date())
        FIN_ENTRENAMIENTO = str((ultima - pd.Timedelta(days=56)).date())
    CORTES = {"entrenamiento": (FECHAS[0], FIN_ENTRENAMIENTO),
              "validacion": (str((pd.Timestamp(FIN_ENTRENAMIENTO) + pd.Timedelta(days=1)).date()), FIN_VALIDACION),
              "prueba": (str((pd.Timestamp(FIN_VALIDACION) + pd.Timedelta(days=1)).date()), FECHAS[-1])}
    print("Cortes:", CORTES)

    conjunto = (F.when(F.col("fecha") <= FIN_ENTRENAMIENTO, "entrenamiento")
                 .when(F.col("fecha") <= FIN_VALIDACION, "validacion").otherwise("prueba"))
    tmc = tm.withColumn("conjunto", conjunto)
    filas_conj = {r.conjunto: int(r["count"]) for r in tmc.groupBy("conjunto").count().collect()}
    print("Filas por conjunto:", filas_conj)

    # Histórico (solo entrenamiento)
    hist = guardar_local(tmc.filter("conjunto = 'entrenamiento'")
                            .withColumn("dow", dia_de(F.col("v"))).withColumn("slot", franja_de(F.col("v")))
                            .groupBy("sentido", "FROM_TO", "dow", "slot")
                            .agg(F.avg("vel_kmh").alias("vel_tipica")), "historico")

    # Muestra
    relevante = F.col("estado") == "trancon"
    for h in HS:
        relevante = relevante | F.coalesce(F.col(f"y_{h}_s") < F.col("vel_libre_ref_kmh") * 0.9, F.lit(False))
    mu = (tmc.withColumn("relevante", relevante.cast("int"))
             .filter((F.col("relevante") == 1) | (F.rand(SEMILLA) < FRAC_FLUIDO))
             .withColumn("peso", F.when(F.col("relevante") == 1, F.lit(1.0)).otherwise(F.lit(1.0 / FRAC_FLUIDO))))
    INFO = ["sentido", "FROM_TO", "v", "fecha", "hora", "conjunto", "estado", "peso", "relevante"]
    pdf = a_pandas(a_numeros(agregar_tipica(mu, hist)), INFO + FEATURES_BASE + TIPICAS + OBJETIVOS, "muestra")
    FEATURES_BASE = [c for c in FEATURES_BASE if c in pdf.columns]
    FEATURES = FEATURES_BASE + TIPICAS
    print(f"Muestra: {len(pdf):,} filas · {pdf.memory_usage(deep=True).sum() / 1024**3:.1f} GB · "
          f"{(time.time() - t0) / 60:.1f} min")
    print(pdf.conjunto.value_counts().to_string())
else:
    print("Modo actualizar: se salta")

**R2 · Entrenamiento y evaluación justa.** Un XGBoost por horizonte. Se detiene cuando deja de mejorar en validación. La evaluación usa solo lo que se sabe al predecir: todas las filas, filas con **trancón ahora** y **hora pico** (días hábiles 5–9 a. m. y 4–8 p. m.).

In [ ]:
# ============================================================
# R2 Entrenamiento (GPU) y evaluación justa
# ============================================================
if MODO == "reentrenar":
    X_all = pdf[FEATURES].to_numpy(dtype=np.float32)
    conj  = pdf["conjunto"].to_numpy()
    peso  = pdf["peso"].to_numpy(dtype=np.float32)
    gc.collect()
    print(f"Matriz de variables: {X_all.shape} · {X_all.nbytes / 1024**3:.1f} GB")

    params = {"eval_metric": "mae", "eta": 0.08, "max_depth": 8, "min_child_weight": 50, "subsample": 0.8,
              "colsample_bytree": 0.8, "seed": SEMILLA, "max_bin": 256, **PARAMS_DISPOSITIVO}
    Matriz = getattr(xgb, "QuantileDMatrix", xgb.DMatrix)
    VARIANTES = ({"mse_peso": ("reg:squarederror", True), "mse_sin_peso": ("reg:squarederror", False),
                  "mae_peso": ("reg:absoluteerror", True), "mae_sin_peso": ("reg:absoluteerror", False)}
                 if PROBAR_VARIANTES else {"mae_peso": ("reg:absoluteerror", True)})

    def entrenar(h, objetivo, usar_peso):
        y = pdf[f"y_{h}_s"].to_numpy(dtype=np.float32)
        i_tr = np.where((conj == "entrenamiento") & ~np.isnan(y))[0]
        i_va = np.where((conj == "validacion") & ~np.isnan(y))[0]
        w_tr, w_va = (peso[i_tr], peso[i_va]) if usar_peso else (None, None)
        dtr = Matriz(X_all[i_tr], label=y[i_tr], weight=w_tr, feature_names=FEATURES)
        dva = (xgb.QuantileDMatrix(X_all[i_va], label=y[i_va], weight=w_va, feature_names=FEATURES, ref=dtr)
               if Matriz is not xgb.DMatrix else
               xgb.DMatrix(X_all[i_va], label=y[i_va], weight=w_va, feature_names=FEATURES))
        bst = xgb.train(dict(params, objective=objetivo), dtr, num_boost_round=3000, evals=[(dva, "validacion")],
                        early_stopping_rounds=100, verbose_eval=False)
        del dtr, dva; gc.collect()
        return Modelo(bst, bst.best_iteration + 1)

    def evaluar_justo(pred, h):
        res = {}
        for c in ["validacion", "prueba"]:
            d = pdf[(pdf.conjunto == c) & pdf[f"y_{h}_s"].notna()]
            p = pd.Series(pred, index=pdf.index).loc[d.index]
            y = d[f"y_{h}_s"]
            ahora = d.estado == "trancon"
            pico = (d.dia_habil == 1) & (d.hora.between(5, 8) | d.hora.between(16, 19))
            res[f"{c}_todas"]         = mae(y, p, d.peso)
            res[f"{c}_trancon_ahora"] = mae(y[ahora], p[ahora], d.peso[ahora])
            res[f"{c}_hora_pico"]     = mae(y[pico], p[pico], d.peso[pico])
        return res

    def pred_referencia(d, h, nombre):
        return d["vel_kmh"] if nombre == "persistencia" else d[f"tipica_{h}"].fillna(d["vel_libre_ref_kmh"])

    filas, modelos, elegidas = [], {}, {}
    for h in HS:
        for ref in ["persistencia", "historico"]:
            filas.append({"horizonte": f"{h} min", "variante": f"ref: {ref}",
                          **evaluar_justo(pred_referencia(pdf, h, ref).to_numpy(), h)})
        mejor = None
        for nombre, (obj, usar_peso) in VARIANTES.items():
            t0 = time.time()
            m = entrenar(h, obj, usar_peso)
            pred = np.clip(m.predict(X_all), 0, None).astype("float32")
            r = {"horizonte": f"{h} min", "variante": nombre, "iter": m.n, **evaluar_justo(pred, h)}
            r["promedio_validacion"] = np.mean([r["validacion_todas"], r["validacion_trancon_ahora"],
                                                r["validacion_hora_pico"]])
            filas.append(r)
            if mejor is None or r["promedio_validacion"] < mejor[0]:
                mejor = (r["promedio_validacion"], nombre, m, pred)
            print(f"{h} min · {nombre}: {m.n} árboles · {time.time() - t0:.0f} s")
        _, elegidas[h], modelos[h], pdf[f"pred_{h}"] = mejor

    justa = pd.DataFrame(filas)
    val_cols = ["validacion_todas", "validacion_trancon_ahora", "validacion_hora_pico"]
    justa["promedio_validacion"] = justa[val_cols].mean(axis=1)
    justa = justa[["horizonte", "variante"] + val_cols +
                  ["prueba_todas", "prueba_trancon_ahora", "prueba_hora_pico", "iter", "promedio_validacion"]]
    print("Variante elegida:", elegidas)
    display(justa.set_index(["horizonte", "variante"]).round(2))
else:
    print("Modo actualizar: se salta")

**R3 · Resultados para el tablero**: referencia del promedio de 30 min, error por hora, por segmento, importancia de variables y un día de ejemplo.

In [ ]:
# ============================================================
# R3 Resultados de desempeño
# ============================================================
if MODO == "reentrenar":
    def a_csv(df, nombre):
        df.to_csv(f"{CARPETA}/{nombre}.csv", index=False)
        print(f"  ✓ {nombre}.csv · {len(df):,} filas")

    # 12.1c Tercera referencia: promedio de los últimos 30 min
    pred_m30 = np.where(np.isnan(pdf["vel_media_30"]), pdf["vel_kmh"], pdf["vel_media_30"])
    filas = []
    for h in HS:
        r = {"horizonte": f"{h} min"}
        for nombre, pred in [("promedio_30min", pred_m30), ("xgboost", pdf[f"pred_{h}"].to_numpy())]:
            res = evaluar_justo(pred, h)
            r.update({f"{nombre}_{k.replace('prueba_', '')}": v for k, v in res.items() if k.startswith("prueba")})
        filas.append(r)
    ref_m30 = pd.DataFrame(filas).round(2)

    # 12.2 Error por hora (prueba, trancón ahora)
    filas = []
    d0 = pdf[pdf.conjunto == "prueba"]
    for h in HS:
        d = d0[d0[f"y_{h}_s"].notna() & (d0.estado == "trancon")]
        for nombre in ["persistencia", "historico", "xgboost"]:
            p = d[f"pred_{h}"] if nombre == "xgboost" else pred_referencia(d, h, nombre)
            g = (pd.DataFrame({"hora": d.hora, "err": (d[f"y_{h}_s"] - p).abs(), "w": d.peso}).dropna()
                   .groupby("hora").apply(lambda x: np.average(x.err, weights=x.w), include_groups=False))
            filas += [{"horizonte": f"{h} min", "modelo": nombre, "hora": int(k), "MAE": float(val)} for k, val in g.items()]
    metricas_hora = pd.DataFrame(filas)

    # 12.3 Importancia de variables (ganancia)
    filas = []
    for h in HS:
        imp = modelos[h].bst.get_score(importance_type="gain")
        tot = sum(imp.values())
        filas += [{"horizonte": f"{h} min", "variable": k, "importancia": val / tot} for k, val in imp.items()]
    importancia = pd.DataFrame(filas)

    # Error por segmento (prueba, trancón ahora)
    m30 = pd.Series(pred_m30, index=pdf.index)
    partes = []
    for h in HS:
        d = d0[d0[f"y_{h}_s"].notna() & (d0.estado == "trancon")]
        y = d[f"y_{h}_s"]
        r = (d.assign(e_xgb=(y - d[f"pred_{h}"]).abs(), e_m30=(y - m30.loc[d.index]).abs(),
                      e_per=(y - d.vel_kmh).abs(), e_his=(y - pred_referencia(d, h, "historico")).abs())
              .groupby(["sentido", "FROM_TO"])
              .agg(filas=("e_xgb", "size"), MAE_xgboost=("e_xgb", "mean"), MAE_promedio30=("e_m30", "mean"),
                   MAE_persistencia=("e_per", "mean"), MAE_historico=("e_his", "mean"))
              .reset_index())
        r["horizonte"] = f"{h} min"
        partes.append(r)
    metricas_segmento = pd.concat(partes).round(2)

    # 12.4 Un día de ejemplo: segmento y día hábil (prueba) con más trancón en la mañana hacia Bogotá
    cand = pdf[(pdf.conjunto == "prueba") & (pdf.sentido == "Occidente - Oriente") &
               (pdf.estado == "trancon") & pdf.hora.between(5, 9) & (pdf.dia_habil == 1)]
    ej = cand.groupby(["FROM_TO", "fecha"]).size().idxmax()
    dia_ej = (tm.filter((F.col("FROM_TO") == ej[0]) & (F.col("fecha") == ej[1])))
    dia_ej = a_pandas(a_numeros(agregar_tipica(dia_ej, hist)), ["sentido", "FROM_TO", "v"] + FEATURES, "dia_ejemplo")
    dia_ej = dia_ej.sort_values("v")
    t_ej = pd.to_datetime(dia_ej.v, unit="s")
    serie = pd.DataFrame({"t": t_ej, "vel_real": dia_ej.vel_kmh.values})
    Xe = dia_ej[FEATURES].to_numpy(dtype=np.float32)
    for h in HS:
        s = pd.DataFrame({"t": t_ej + pd.Timedelta(minutes=MINUTOS[h]),
                          f"pred_{h}": np.clip(modelos[h].predict(Xe), 0, None),
                          f"promedio30_{h}": dia_ej.vel_media_30.values,
                          f"persistencia_{h}": dia_ej.vel_kmh.values,
                          f"historico_{h}": dia_ej[f"tipica_{h}"].values})
        serie = serie.merge(s, on="t", how="outer")
    serie = serie[serie.t.dt.strftime("%Y-%m-%d") == ej[1]].sort_values("t")
    serie["t"] = serie.t.dt.strftime("%Y-%m-%d %H:%M")
    serie["FROM_TO"], serie["fecha"] = ej[0], ej[1]

    print("Resultados:")
    a_csv(justa.round(3), "metricas_justas")
    a_csv(ref_m30, "referencia_promedio30")
    a_csv(metricas_hora.round(3), "metricas_hora")
    a_csv(importancia.round(5), "importancia_variables")
    a_csv(metricas_segmento, "metricas_segmento")
    a_csv(serie.round(2), "serie_ejemplo")
    print("\nPrueba (MAE km/h):")
    display(ref_m30.set_index("horizonte"))
else:
    print("Modo actualizar: se salta")

**R4 · Guardar el modelo en Azure**: los 3 modelos, la lista de variables, el número de árboles, el histórico y la ficha técnica. Va a `modelos/actual/` (lo que usa el modo actualizar) y una copia a `modelos/historial/AAAAMMDD_HHMM/`.

In [ ]:
# ============================================================
# R4 Guardar el modelo
# ============================================================
if MODO == "reentrenar":
    def txt_periodo(a, b):
        MES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        return f"{a.day} {MES[a.month - 1]} – {b.day} {MES[b.month - 1]} {b.year}"

    lim = tm.agg(F.min("v").alias("a"), F.max("v").alias("b")).first()
    ficha = pd.DataFrame([
        ("corredor", "Corredor", "Autopista Sur (Bogotá – Soacha – Granada)"),
        ("fuente", "Fuente", "Waze for Cities (jams)"),
        ("desde", "Primer dato", pd.to_datetime(lim.a, unit="s").strftime("%Y-%m-%d %H:%M")),
        ("hasta", "Último dato", pd.to_datetime(lim.b, unit="s").strftime("%Y-%m-%d %H:%M")),
        ("filas_tabla_modelo", "Filas tabla de modelado", sum(filas_conj.values())),
        ("filas_muestra", "Filas muestra de entrenamiento", len(pdf)),
        ("segmentos_activos", "Segmentos monitoreados", vl_act.count()),
        ("filas_entrenamiento", f"Filas entrenamiento ({txt_periodo(*CORTES['entrenamiento'])})", filas_conj.get("entrenamiento", 0)),
        ("filas_validacion", f"Filas validación ({txt_periodo(*CORTES['validacion'])})", filas_conj.get("validacion", 0)),
        ("filas_prueba", f"Filas prueba ({txt_periodo(*CORTES['prueba'])})", filas_conj.get("prueba", 0)),
        ("objetivo", "Variable objetivo", "velocidad promedio de 15 min alrededor del horizonte (km/h)"),
        ("modelo", "Modelo", f"XGBoost {xgb.__version__} ({'GPU' if GPU else 'CPU'}), "
                             f"{'error absoluto, con pesos' if set(elegidas.values()) == {'mae_peso'} else elegidas}"),
    ] + [(f"arboles_{h}", f"Árboles modelo {h} min", modelos[h].n) for h in HS],
        columns=["clave", "etiqueta", "valor"])

    SELLO = time.strftime("%Y%m%d_%H%M")
    LOCAL_MOD = f"{CONTENT}/modelo_nuevo"
    os.makedirs(LOCAL_MOD, exist_ok=True)
    for h in HS:
        modelos[h].bst.save_model(f"{LOCAL_MOD}/modelo_xgb_{h}min.json")
    meta = {"features": FEATURES, "arboles": {h: modelos[h].n for h in HS}, "variante": elegidas,
            "cortes": CORTES, "entrenado": SELLO, "xgboost": xgb.__version__,
            "prueba": ref_m30.set_index("horizonte").to_dict()}
    json.dump(meta, open(f"{LOCAL_MOD}/meta.json", "w"), ensure_ascii=False, indent=2)
    ficha.to_csv(f"{LOCAL_MOD}/ficha_tecnica.csv", index=False)

    for destino in [f"{MODELOS}/historial/{SELLO}", ACTUAL]:
        borrar(destino)
        for nombre in os.listdir(LOCAL_MOD):
            fs_de(destino).copyFromLocalFile(False, True, Path(f"file://{LOCAL_MOD}/{nombre}"),
                                             Path(f"{destino}/{nombre}"))
        hist.write.mode("overwrite").parquet(f"{destino}/historico")
        print("Modelo guardado en", destino)
    del X_all; gc.collect()
else:
    print("Modo actualizar: se salta")

## P. Predicción y datos del tablero (siempre)

**P1 · Cargar el modelo.** En modo actualizar se baja de `modelos/actual/`; en modo reentrenar ya está en memoria.

In [ ]:
# ============================================================
# P1 Modelo, histórico y ficha técnica
# ============================================================
if MODO == "actualizar":
    if not existe(ACTUAL):
        raise SystemExit("No hay modelo guardado: corre este cuaderno una vez en Colab con MODO = 'reentrenar'")
    LOCAL_MOD = f"{CONTENT}/modelo_actual"
    os.makedirs(LOCAL_MOD, exist_ok=True)
    for nombre in ["meta.json", "ficha_tecnica.csv"] + [f"modelo_xgb_{h}min.json" for h in HS]:
        fs_de(ACTUAL).copyToLocalFile(False, Path(f"{ACTUAL}/{nombre}"), Path(f"file://{LOCAL_MOD}/{nombre}"), True)
    meta = json.load(open(f"{LOCAL_MOD}/meta.json"))
    FEATURES = meta["features"]
    FEATURES_BASE = [c for c in FEATURES if c not in TIPICAS]
    modelos = {}
    for h in HS:
        b = xgb.Booster()
        b.load_model(f"{LOCAL_MOD}/modelo_xgb_{h}min.json")
        b.set_param({"device": "cpu"})
        modelos[h] = Modelo(b, meta["arboles"][h])
    hist = spark.read.parquet(f"{ACTUAL}/historico")
    ficha = pd.read_csv(f"{LOCAL_MOD}/ficha_tecnica.csv", dtype=str)
    print(f"Modelo entrenado el {meta['entrenado']} · cortes: {meta['cortes']}")
else:
    print("Modelo recién entrenado, en memoria")

**P2 · Último día con datos**: velocidad observada, típica y predicha a +15, +30 y +60 min por segmento y ventana (`dia_monitoreo`), la predicción desde la última ventana, el perfil típico del corredor y la ficha del día.

In [ ]:
# ============================================================
# P2 Archivos del tablero que cambian cada día
# ============================================================
DIA = FECHAS[-1]
base = (tm.filter(F.col("fecha") == DIA)
          .withColumn("dow0", dia_de(F.col("v"))).withColumn("slot0", franja_de(F.col("v")))
          .join(F.broadcast(hist.select("sentido", "FROM_TO", F.col("dow").alias("dow0"),
                                        F.col("slot").alias("slot0"), F.col("vel_tipica").alias("tip"))),
                ["sentido", "FROM_TO", "dow0", "slot0"], "left"))
cols_dia = ["sentido", "FROM_TO", "v", "estado", "hora", "tip"] + FEATURES + OBJETIVOS
d = a_pandas(a_numeros(agregar_tipica(base, hist)), cols_dia, "dia").sort_values(["v", "sentido", "FROM_TO"])
Xd = d[FEATURES].to_numpy(dtype=np.float32)
for h in HS:
    d[f"p{h}"] = np.clip(modelos[h].predict(Xd), 0, None)
d["hhmm"] = pd.to_datetime(d.v, unit="s").dt.strftime("%H:%M")
print(f"Último día: {DIA} · datos de {d.hhmm.min()} a {d.hhmm.max()} · "
      f"ventanas con trancón: {(d.estado == 'trancon').sum():,}")

# ---- 1. dia_monitoreo.csv (se acorta desde la mañana si pesa demasiado) ----
def armar_dia(desde, hasta):
    x = d[d.hora.between(desde, hasta)]
    out = pd.DataFrame({"hora": x.hhmm.values, "FROM_TO": x.FROM_TO.values, "estado": x.estado.values,
                        "vel": x.vel_kmh.round(1).values, "tip": x.tip.round(1).values})
    for h in HS:
        out[f"p{h}"] = x[f"p{h}"].round(1).values
        out[f"r{h}"] = x[f"y_{h}_s"].round(1).values
    return out

hasta = int(min(HORA_HASTA, d.hora.max()))
desde = int(min(HORA_DESDE, hasta))
while True:
    dia = armar_dia(desde, hasta)
    dia.to_csv(f"{CARPETA}/dia_monitoreo.csv", index=False)
    tam = os.path.getsize(f"{CARPETA}/dia_monitoreo.csv") / 1024**2
    if tam <= MAX_MB_DIA or hasta - desde <= 8:
        break
    desde += 1
print(f"  ✓ dia_monitoreo.csv · {len(dia):,} filas · {desde}:00 a {hasta}:55 · {tam:.1f} MB")

# ---- 2. dia_info.csv ----
DIAS  = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
f = pd.Timestamp(DIA)
pd.DataFrame([{"fecha": DIA, "fecha_texto": f"{DIAS[f.dayofweek]} {f.day} de {MESES[f.month - 1]} de {f.year}",
               "dia_semana": f.dayofweek, "hora_desde": desde, "hora_hasta": hasta}]
            ).to_csv(f"{CARPETA}/dia_info.csv", index=False)
print("  ✓ dia_info.csv")

# ---- 3. predicciones_ultima_ventana.csv ----
v_ult = int(d.v.max())
u = d[d.v == v_ult].copy()
t_ult = pd.to_datetime(v_ult, unit="s")
pred_ult = u[["sentido", "FROM_TO", "abs_ini_m", "vel_libre_ref_kmh", "estado", "vel_kmh"]].copy()
pred_ult["km"] = (pred_ult.abs_ini_m / 1000).round(2)
for h in HS:
    pred_ult[f"pred_{h}min"] = u[f"p{h}"].round(1).values
    pred_ult[f"hora_{h}min"] = (t_ult + pd.Timedelta(minutes=MINUTOS[h])).strftime("%Y-%m-%d %H:%M")
pred_ult["ventana_base"] = t_ult.strftime("%Y-%m-%d %H:%M")
pred_ult.sort_values(["sentido", "km"]).to_csv(f"{CARPETA}/predicciones_ultima_ventana.csv", index=False)
print(f"  ✓ predicciones_ultima_ventana.csv · última ventana {t_ult:%Y-%m-%d %H:%M} · {len(pred_ult)} segmentos")

# ---- 4. perfil_tipico.csv (velocidad típica del corredor) ----
perfil = (hist.join(vl_act.select("sentido", "FROM_TO", "longitud_m"), ["sentido", "FROM_TO"])
              .filter("longitud_m > 0 AND vel_tipica > 0")
              .groupBy("sentido", "dow", "slot")
              .agg((F.sum("longitud_m") / F.sum(F.col("longitud_m") / F.col("vel_tipica"))).alias("vel_tipica_kmh"))
              .toPandas())
perfil["hora"] = ((perfil.slot * 5 // 60).astype(int).astype(str).str.zfill(2) + ":" +
                  (perfil.slot * 5 % 60).astype(int).astype(str).str.zfill(2))
(perfil.rename(columns={"dow": "dia_semana"})[["sentido", "dia_semana", "hora", "vel_tipica_kmh"]]
       .round(2).sort_values(["sentido", "dia_semana", "hora"])
       .to_csv(f"{CARPETA}/perfil_tipico.csv", index=False))
print(f"  ✓ perfil_tipico.csv · {len(perfil):,} filas")

# ---- 6. segmentos_atributos.csv (ciudad y vía para los filtros del tablero) ----
RUTA_ATR = f"{V3}/atributos_segmento"
if existe(RUTA_ATR):
    (spark.read.parquet(RUTA_ATR).filter("activo")
          .select("FROM_TO", "sentido", "city", "NomVia2",
                  F.round(F.col("longitud_m") / 1000, 3).alias("longitud_km"),
                  F.round(F.col("abs_ini_m") / 1000, 3).alias("km"))
          .toPandas().to_csv(f"{CARPETA}/segmentos_atributos.csv", index=False))
    print("  ✓ segmentos_atributos.csv")
else:
    print("  ✗ segmentos_atributos.csv: falta v3/atributos_segmento (corre la celda 1.3 del cuaderno 01)")

# ---- 5. ficha_tecnica.csv (con el último dato actualizado) ----
ficha.loc[ficha.clave == "hasta", "valor"] = t_ult.strftime("%Y-%m-%d %H:%M")
ficha.to_csv(f"{CARPETA}/ficha_tecnica.csv", index=False)
print("  ✓ ficha_tecnica.csv")

**P3 · Empaquetar y descargar.** Este `.zip` es el que se sube a GitHub (carpeta `tablero/datos/`) o se adjunta en el chat.

In [ ]:
# ============================================================
# P3 Paquete para el tablero
# ============================================================
import zipfile

ruta_zip = f"{CONTENT}/tablero_{DIA}{'_reentreno' if MODO == 'reentrenar' else ''}.zip"
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for nombre in sorted(os.listdir(CARPETA)):
        z.write(f"{CARPETA}/{nombre}", nombre)
        print(f"  {nombre:<34} {os.path.getsize(f'{CARPETA}/{nombre}') / 1024**2:6.2f} MB")
print(f"\n{ruta_zip} · {os.path.getsize(ruta_zip) / 1024**2:.1f} MB")

# Copia en Azure: el cuaderno 04 la toma de aquí para publicar en GitHub
RUTA_TABLERO = f"{V3}/tablero"
for nombre in os.listdir(CARPETA):
    fs_de(RUTA_TABLERO).copyFromLocalFile(False, True, Path(f"file://{CARPETA}/{nombre}"),
                                          Path(f"{RUTA_TABLERO}/{nombre}"))
print("CSV copiados a", RUTA_TABLERO, "→ sigue el cuaderno 04_PUBLICAR_TABLERO")

for nombre in os.listdir(TMP_LOCAL):
    if nombre != "spark":
        shutil.rmtree(f"{TMP_LOCAL}/{nombre}", ignore_errors=True)
if PROJECT_DIR:                                      # copia en Google Drive: Waze_GD/Salidas
    shutil.copy(ruta_zip, PATH_SALIDAS)
    print("Copia en Drive:", f"{PATH_SALIDAS}/{os.path.basename(ruta_zip)}")
if EN_COLAB:
    from google.colab import files
    files.download(ruta_zip)